# Notebook Overview — Prepare Video Data

## Purpose

This notebook prepares the NExT-QA benchmark dataset for iterative Retrieval-Augmented Generation (RAG) Video Question Answering (VideoQA) experimentation. The workflow configures the runtime environment, verifies required dataset resources, reconstructs and extracts video archives, organizes dataset files, and validates the dataset structure required for downstream preprocessing, embedding generation, retrieval, and inference workflows.

## Inputs

* NExT-QA multipart video archive files stored in Google Drive
  * NExTVideo.z01
  * NExTVideo.z02
  * NExTVideo.z03
  * NExTVideo.z04
  * NExTVideo.z05
  * NExTVideo.z06
  * NExTVideo.zip
* NExT-QA question-answer annotation files
  * train.csv
  * val.csv
  * test.csv
* NExT-QA metadata resources
  * map_vid_vidorID.json
* User configuration settings
* Project configuration modules

## Outputs

* Verified NExT-QA dataset directory structure
* Reconstructed and extracted video dataset
* Validated question-answer annotation files
* Verified metadata resources
* Dataset statistics and verification summaries
* Runtime environment configuration information

## Processing Workflow

* Configure runtime environment and project settings
* Mount Google Drive and verify dataset resources
* Copy video archives to local storage
* Reconstruct and extract the NExT-QA video archive
* Validate video, annotation, and metadata resources
* Display dataset statistics and verification summaries

## Notes

* This notebook focuses on dataset preparation and validation only.
* The NExT-QA benchmark serves as the primary VideoQA dataset for this project.
* Video archives are copied to local Colab storage and extracted locally to improve reliability and performance.
* Frame extraction, clip generation, embedding creation, vector indexing, retrieval, and inference workflows are performed in later notebooks.
* Video archive reconstruction and extraction may require significant storage space and execution time depending on the runtime environment.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True

import os
from google.colab import userdata

REPO_NAME = "iterative-video-rag"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository files...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --no-cone

    !git sparse-checkout set \
        src/iterative_rag_config.py \
        datasets/NExT-QA/questions \
        datasets/NExT-QA/metadata

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "src/iterative_rag_config.py",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nAvailable files:")
    !find src datasets/NExT-QA -maxdepth 3 -type f | sort



### 🔷 Step 2 — Import Project Configuration

* Import centralized project configuration settings and constants from the project configuration module.
* Load repository paths, dataset directories, and runtime configuration parameters used throughout the VideoQA workflow.
* Initialize reusable configuration values shared across notebook execution stages.
* Verify that required configuration resources are accessible and correctly loaded before continuing.
* Optionally display configuration settings and resolved paths when `VERBOSE=True`.

In [ ]:
# ============================================================
# Step 2: Import Project Configuration
# ============================================================

# ------------------------------------------------------------
# Import Centralized Project Configuration Values
# ------------------------------------------------------------

try:

    from src.iterative_rag_config import *

except ModuleNotFoundError as error:

    raise ModuleNotFoundError(
        "Project configuration could not be imported. "
        "Verify that the repository was cloned correctly and that "
        "'src/iterative_rag_config.py' exists."
    ) from error


# ------------------------------------------------------------
# Verify Required Configuration Values
# ------------------------------------------------------------

required_config_values = [
    "BASE_DIR",
    "DATASETS_DIR",
    "DATASET_CONFIG",
]

missing_config_values = [
    name
    for name in required_config_values
    if name not in globals()
]

if missing_config_values:

    raise ValueError(
        "Missing required configuration values: "
        + ", ".join(missing_config_values)
    )


# ------------------------------------------------------------
# Display Configuration Summary
# ------------------------------------------------------------

print("Project configuration imported successfully.")

if VERBOSE:
    print(f"BASE_DIR:      {BASE_DIR}")
    print(f"DATASETS_DIR:  {DATASETS_DIR}")

    print("\nConfigured Datasets:")

    for dataset_name in DATASET_CONFIG:
        print(f"  - {dataset_name}")



### 🔷 Step 3 — Mount Google Drive

* Mount Google Drive to access benchmark dataset archives and supporting resources.
* Verify that Google Drive was mounted successfully and is accessible from the notebook runtime.
* Configure access to the NExT-QA dataset storage location.
* Confirm that required dataset resources are available before proceeding with extraction and validation steps.
* Optionally display mounted paths and available dataset files when `VERBOSE=True`.

In [ ]:
# ============================================================
# Step 3: Mount Google Drive
# ============================================================

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
import os

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    if VERBOSE:
        print("Mounting Google Drive...")

    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    if VERBOSE:
        print("Google Drive is already mounted.")


# ------------------------------------------------------------
# Verify Google Drive Access
# ------------------------------------------------------------

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    raise FileNotFoundError(
        "Google Drive mount point was not found."
    )

drive_root = os.path.join(
    GOOGLE_DRIVE_MOUNT,
    "MyDrive"
)

if not os.path.exists(drive_root):

    raise FileNotFoundError(
        "Unable to access Google Drive root directory."
    )

print("Google Drive mounted successfully.")


# ------------------------------------------------------------
# Display Drive Information
# ------------------------------------------------------------

if VERBOSE:
    print(f"\nDrive Root: {drive_root}")
    print("\nTop-Level Google Drive Folders:")

    try:
        drive_items = sorted(os.listdir(drive_root))
        for item in drive_items[:20]:
            print(f"  {item}")
        if len(drive_items) > 20:
            print(
                f"\n... and "
                f"{len(drive_items) - 20} additional items"
            )

    except Exception as error:

        print(
            f"Unable to list Google Drive contents: {error}"
        )



### 🔷 Step 4 — Verify NExT-QA Dataset Resources

* Configure the Google Drive location containing the NExT-QA dataset archive files.
* Verify that all required multipart video archive files are present.
* Display file sizes for the available dataset resources.
* Stop notebook execution if any required archive files are missing.
* Prepare validated archive paths for later extraction.

In [ ]:
# ============================================================
# Step 4: Verify NExT-QA Dataset Resources
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Configure Google Drive Dataset Source Directory
# ------------------------------------------------------------

DRIVE_DATASET_DIR = Path(drive_root) / "VideoQA_Project" / "NExT-QA"

if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"NExT-QA Google Drive folder not found: {DRIVE_DATASET_DIR}"
    )


# ------------------------------------------------------------
# Define Required Archive Files
# ------------------------------------------------------------

required_archive_files = [
    "NExTVideo.z01",
    "NExTVideo.z02",
    "NExTVideo.z03",
    "NExTVideo.z04",
    "NExTVideo.z05",
    "NExTVideo.z06",
    "NExTVideo.zip",
]


# ------------------------------------------------------------
# Verify Required Archive Files
# ------------------------------------------------------------

missing_archive_files = []

print("Checking NExT-QA video archive files...")

for filename in required_archive_files:

    file_path = DRIVE_DATASET_DIR / filename

    if not file_path.exists():
        missing_archive_files.append(filename)

    elif VERBOSE:
        file_size_gb = file_path.stat().st_size / (1024 ** 3)
        print(f"  FOUND: {filename:<16} {file_size_gb:8.2f} GB")


if missing_archive_files:
    raise FileNotFoundError(
        "Missing required NExT-QA archive files: "
        + ", ".join(missing_archive_files)
    )


# ------------------------------------------------------------
# Display Verification Summary
# ------------------------------------------------------------

print("All required NExT-QA video archive files were found.")

if VERBOSE:

    print(f"\nGoogle Drive Dataset Directory:")
    print(f"  {DRIVE_DATASET_DIR}")

    print("\nAvailable files:")

    for file_path in sorted(DRIVE_DATASET_DIR.iterdir()):
        if file_path.is_file():
            file_size_mb = file_path.stat().st_size / (1024 ** 2)
            print(f"  {file_path.name:<28} {file_size_mb:10.2f} MB")



### 🔷 Step 5 — Copy NExT-QA Archive Files to Local Storage

* Create a local archive workspace within the Colab runtime environment.
* Copy the NExT-QA multipart archive files from Google Drive to local storage.
* Verify that all required archive files were copied successfully.
* Display archive file sizes and storage utilization information when `VERBOSE=True`.
* Skip file copies when valid local archive files already exist.
* Prepare local archive resources for archive reconstruction and extraction in later steps.

In [ ]:
# ============================================================
# Step 5: Copy NExT-QA Archive Files to Local Storage
# ============================================================

import shutil
import time
from pathlib import Path

# ------------------------------------------------------------
# Configure Local Archive Directory
# ------------------------------------------------------------

LOCAL_ARCHIVE_DIR = DATASETS_DIR / "NExT-QA" / "archives"

LOCAL_ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Define Required Archive Files
# ------------------------------------------------------------

required_archive_files = [
    "NExTVideo.z01",
    "NExTVideo.z02",
    "NExTVideo.z03",
    "NExTVideo.z04",
    "NExTVideo.z05",
    "NExTVideo.z06",
    "NExTVideo.zip",
]

# ------------------------------------------------------------
# Copy Archive Files to Local Storage
# ------------------------------------------------------------

print("Copying NExT-QA archive files to local storage...")

copied_files = []
skipped_files = []

for filename in required_archive_files:

    source_path = DRIVE_DATASET_DIR / filename
    destination_path = LOCAL_ARCHIVE_DIR / filename

    source_size = source_path.stat().st_size
    source_size_gb = source_size / (1024 ** 3)

    copy_required = True

    if destination_path.exists():

        destination_size = destination_path.stat().st_size

        if destination_size == source_size:

            copy_required = False
            skipped_files.append(filename)

    if copy_required:

        print(
            f"  Copying: {filename} "
            f"({source_size_gb:.2f} GB)"
        )

        start_time = time.time()

        shutil.copy2(
            source_path,
            destination_path,
        )

        elapsed_time = time.time() - start_time

        copied_files.append(filename)

        print(
            f"  Completed: {filename} "
            f"({elapsed_time:.1f} seconds)"
        )

    else:

        print(
            f"  Skipping: {filename} "
            f"(local copy already exists)"
        )

# ------------------------------------------------------------
# Verify Local Archive Files
# ------------------------------------------------------------

missing_local_files = []
size_mismatch_files = []

for filename in required_archive_files:

    source_path = DRIVE_DATASET_DIR / filename
    local_file = LOCAL_ARCHIVE_DIR / filename

    if not local_file.exists():

        missing_local_files.append(filename)

    elif local_file.stat().st_size != source_path.stat().st_size:

        size_mismatch_files.append(filename)

if missing_local_files:

    raise FileNotFoundError(
        "Missing local archive files: "
        + ", ".join(missing_local_files)
    )

if size_mismatch_files:

    raise ValueError(
        "Local archive file size mismatch detected: "
        + ", ".join(size_mismatch_files)
    )

# ------------------------------------------------------------
# Display Copy Summary
# ------------------------------------------------------------

print("Local archive verification complete.")

print(f"Files copied:  {len(copied_files)}")
print(f"Files skipped: {len(skipped_files)}")

# ------------------------------------------------------------
# Display Archive Information
# ------------------------------------------------------------

if VERBOSE:

    total_archive_size_gb = sum(
        file_path.stat().st_size
        for file_path in LOCAL_ARCHIVE_DIR.iterdir()
        if file_path.is_file()
    ) / (1024 ** 3)

    print(f"\nLocal Archive Directory:")
    print(f"  {LOCAL_ARCHIVE_DIR}")

    print(f"\nTotal Archive Size:")
    print(f"  {total_archive_size_gb:.2f} GB")

    print("\nLocal Archive Files:")

    for file_path in sorted(LOCAL_ARCHIVE_DIR.iterdir()):

        if file_path.is_file():

            file_size_gb = (
                file_path.stat().st_size
                / (1024 ** 3)
            )

            print(
                f"  {file_path.name:<16} "
                f"{file_size_gb:8.2f} GB"
            )



### 🔷 Step 6 — Build Combined NExT-QA Archive

* Reconstruct the original NExT-QA video archive from the locally copied multipart archive files.
* Concatenate all archive segments in the correct sequence to create a single combined ZIP archive.
* Verify that the combined archive was created successfully and has a valid file size.
* Skip archive reconstruction when a valid combined archive already exists.
* Display archive creation statistics and storage utilization information when `VERBOSE=True`.
* Prepare the combined archive for local extraction in the next processing step.


In [ ]:
# ============================================================
# Step 6: Build Combined NExT-QA Archive
# ============================================================

import subprocess
import time
from pathlib import Path

# ------------------------------------------------------------
# Configure Combined Archive Path
# ------------------------------------------------------------

COMBINED_ARCHIVE_PATH = LOCAL_ARCHIVE_DIR / "NExTVideo_combined.zip"
SPLIT_ARCHIVE_PATH = LOCAL_ARCHIVE_DIR / "NExTVideo.zip"

# ------------------------------------------------------------
# Verify Required Archive Parts
# ------------------------------------------------------------

required_archive_files = [
    "NExTVideo.z01",
    "NExTVideo.z02",
    "NExTVideo.z03",
    "NExTVideo.z04",
    "NExTVideo.z05",
    "NExTVideo.z06",
    "NExTVideo.zip",
]

for filename in required_archive_files:

    file_path = LOCAL_ARCHIVE_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required archive part not found: {file_path}"
        )

# ------------------------------------------------------------
# Remove Invalid Combined Archive if Present
# ------------------------------------------------------------

if COMBINED_ARCHIVE_PATH.exists():

    print("Removing existing combined archive before rebuild.")

    COMBINED_ARCHIVE_PATH.unlink()

# ------------------------------------------------------------
# Convert Split ZIP Archive into Single ZIP Archive
# ------------------------------------------------------------

print("Building combined NExT-QA archive using zip split-archive conversion...")
print("This may take several minutes.")

start_time = time.time()

convert_command = [
    "zip",
    "-s",
    "0",
    str(SPLIT_ARCHIVE_PATH),
    "--out",
    str(COMBINED_ARCHIVE_PATH),
]

result = subprocess.run(
    convert_command,
    cwd=str(LOCAL_ARCHIVE_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

elapsed_time = time.time() - start_time

if result.returncode != 0:

    raise RuntimeError(
        "Failed to build combined NExT-QA archive.\n\n"
        f"Command: {' '.join(convert_command)}\n\n"
        f"Elapsed Time: {elapsed_time:.1f} seconds\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

print(
    "Combined archive created "
    f"({elapsed_time:.1f} seconds)."
)

# ------------------------------------------------------------
# Verify Combined Archive
# ------------------------------------------------------------

if not COMBINED_ARCHIVE_PATH.exists():

    raise FileNotFoundError(
        f"Combined archive was not created: {COMBINED_ARCHIVE_PATH}"
    )

combined_size_gb = COMBINED_ARCHIVE_PATH.stat().st_size / (1024 ** 3)

print("Combined archive verification complete.")

if VERBOSE:

    print(f"\nCombined Archive Path:")
    print(f"  {COMBINED_ARCHIVE_PATH}")

    print(f"\nCombined Archive Size:")
    print(f"  {combined_size_gb:.2f} GB")



### 🔷 Step 7 — Extract NExT-QA Video Archive

* Create the local NExT-QA video directory within the project dataset workspace.
* Extract the combined NExT-QA video archive into local Colab storage.
* Preserve the original NExT-QA video folder structure during extraction.
* Skip extraction when extracted video files are already present.
* Verify that extraction completed without archive errors.
* Prepare the extracted video dataset for structure validation in the next step.


In [ ]:
# ============================================================
# Step 7: Extract NExT-QA Video Archive
# ============================================================

import subprocess
import time
from pathlib import Path

# ------------------------------------------------------------
# Configure Local Video Directory
# ------------------------------------------------------------

NEXTQA_DATASET_DIR = DATASETS_DIR / "NExT-QA"
NEXTQA_VIDEOS_DIR = NEXTQA_DATASET_DIR / "videos"

NEXTQA_VIDEOS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Check Whether Videos Are Already Extracted
# ------------------------------------------------------------

existing_video_files = list(NEXTQA_VIDEOS_DIR.rglob("*.mp4"))

if existing_video_files:

    print("NExT-QA videos already appear to be extracted.")
    print(f"Existing video files found: {len(existing_video_files)}")

else:

    # ------------------------------------------------------------
    # Verify Combined Archive Exists
    # ------------------------------------------------------------

    if not COMBINED_ARCHIVE_PATH.exists():

        raise FileNotFoundError(
            f"Combined archive not found: {COMBINED_ARCHIVE_PATH}"
        )

    # ------------------------------------------------------------
    # Extract Combined Archive
    # ------------------------------------------------------------

    print("Extracting combined NExT-QA video archive...")
    print("This may take several minutes.")

    start_time = time.time()

    extract_command = [
        "unzip",
        "-qo",
        str(COMBINED_ARCHIVE_PATH),
        "-d",
        str(NEXTQA_VIDEOS_DIR),
    ]

    result = subprocess.run(
        extract_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    elapsed_time = time.time() - start_time

    if result.returncode != 0:

        raise RuntimeError(
            "NExT-QA video archive extraction failed.\n\n"
            f"Command: {' '.join(extract_command)}\n\n"
            f"Elapsed Time: {elapsed_time:.1f} seconds\n\n"
            f"Error Output:\n{result.stderr}"
        )

    print(
        "NExT-QA video archive extraction complete "
        f"({elapsed_time:.1f} seconds)."
    )

# ------------------------------------------------------------
# Verify Extracted Video Files
# ------------------------------------------------------------

video_files = list(NEXTQA_VIDEOS_DIR.rglob("*.mp4"))

if not video_files:

    raise FileNotFoundError(
        "No MP4 files were found after archive extraction."
    )

print("NExT-QA video extraction verified.")
print(f"Video files found: {len(video_files)}")

# ------------------------------------------------------------
# Display Extraction Summary
# ------------------------------------------------------------

if VERBOSE:

    total_video_size_gb = sum(
        file_path.stat().st_size
        for file_path in video_files
    ) / (1024 ** 3)

    video_subdirs = sorted(
        path for path in NEXTQA_VIDEOS_DIR.rglob("*")
        if path.is_dir()
    )

    print(f"\nLocal Dataset Directory:")
    print(f"  {NEXTQA_DATASET_DIR}")

    print(f"\nLocal Video Directory:")
    print(f"  {NEXTQA_VIDEOS_DIR}")

    print(f"\nExtracted Video Size:")
    print(f"  {total_video_size_gb:.2f} GB")

    print(f"\nVideo Subdirectories Found:")
    print(f"  {len(video_subdirs)}")

    print("\nSample Video Files:")

    for file_path in sorted(video_files)[:10]:

        relative_path = file_path.relative_to(NEXTQA_VIDEOS_DIR)
        print(f"  {relative_path}")



### 🔷 Step 8 — Verify Question and Metadata Resources

* Verify the presence of required NExT-QA question-answer annotation files.
* Verify the presence of required NExT-QA metadata resources.
* Validate accessibility of training, validation, and test dataset files.
* Display annotation file sizes and basic dataset statistics.
* Confirm that all required resources are available for downstream VideoQA experimentation.
* Stop notebook execution if required annotation or metadata files are missing.


In [ ]:
# ============================================================
# Step 8: Verify Question and Metadata Resources
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Configure NExT-QA Dataset Resource Directories
# ------------------------------------------------------------

#NEXTQA_DATASET_DIR = DATASETS_DIR / "NExT-QA"
NEXTQA_QUESTIONS_DIR = DATASET_CONFIG["NExT-QA"]["questions_dir"]
NEXTQA_METADATA_DIR  = DATASET_CONFIG["NExT-QA"]["metadata_dir"]

NEXTQA_QUESTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NEXTQA_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Define Required Question and Metadata Files
# ------------------------------------------------------------

required_question_files = [
    "train.csv",
    "val.csv",
    "test.csv",
]

required_metadata_files = [
    "map_vid_vidorID.json",
]

# ------------------------------------------------------------
# Verify Required Files
# ------------------------------------------------------------

missing_question_files = []
missing_metadata_files = []

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename

    if not file_path.exists():

        missing_question_files.append(filename)

for filename in required_metadata_files:

    file_path = NEXTQA_METADATA_DIR / filename

    if not file_path.exists():

        missing_metadata_files.append(filename)

if missing_question_files:

    raise FileNotFoundError(
        "Missing required question files: "
        + ", ".join(missing_question_files)
    )

if missing_metadata_files:

    raise FileNotFoundError(
        "Missing required metadata files: "
        + ", ".join(missing_metadata_files)
    )

# ------------------------------------------------------------
# Load Question Files and Count Rows
# ------------------------------------------------------------

question_counts = {}

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename

    dataframe = pd.read_csv(file_path)

    question_counts[filename] = len(dataframe)

# ------------------------------------------------------------
# Display Verification Summary
# ------------------------------------------------------------

print("NExT-QA question and metadata resources verified.")

print("\nQuestion Files:")

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename
    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"  {filename:<10} "
        f"{question_counts[filename]:>8} rows "
        f"{file_size_mb:>8.2f} MB"
    )

print("\nMetadata Files:")

for filename in required_metadata_files:

    file_path = NEXTQA_METADATA_DIR / filename
    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"  {filename:<32} "
        f"{file_size_mb:>8.2f} MB"
    )

# ------------------------------------------------------------
# Display Optional Dataset Details
# ------------------------------------------------------------

if VERBOSE:

    total_questions = sum(question_counts.values())

    print(f"\nTotal Questions: {total_questions}")

    print("\nDataset Resource Directories:")
    print(f"  Questions: {NEXTQA_QUESTIONS_DIR}")
    print(f"  Metadata:  {NEXTQA_METADATA_DIR}")



### 🔷 Step 9 — Display Dataset Statistics and Verification Summary

* Summarize the results of dataset preparation, archive reconstruction, extraction, and resource validation.
* Report video dataset statistics including extracted video counts, directory counts, and storage utilization.
* Report question-answer annotation statistics for the training, validation, and test partitions.
* Summarize available metadata resources and dataset configuration information.
* Confirm overall NExT-QA dataset readiness for downstream VideoQA experimentation.
* Provide a consolidated verification report for use by subsequent notebooks.


In [ ]:
# ============================================================
# Step 9: Display Dataset Statistics and Verification Summary
# ============================================================

# ------------------------------------------------------------
# Collect Video Dataset Statistics
# ------------------------------------------------------------

NEXTQA_CONFIG = DATASET_CONFIG["NExT-QA"]

NEXTQA_DATASET_DIR = NEXTQA_CONFIG["dataset_dir"]
NEXTQA_VIDEOS_DIR = NEXTQA_CONFIG["videos_dir"]
NEXTQA_QUESTIONS_DIR = NEXTQA_CONFIG["questions_dir"]
NEXTQA_METADATA_DIR = NEXTQA_CONFIG["metadata_dir"]

video_files = sorted(
    NEXTQA_VIDEOS_DIR.rglob("*.mp4")
)

video_subdirs = sorted(
    path for path in NEXTQA_VIDEOS_DIR.rglob("*")
    if path.is_dir()
)

total_video_size_gb = sum(
    file_path.stat().st_size
    for file_path in video_files
) / (1024 ** 3)

# ------------------------------------------------------------
# Collect Question Dataset Statistics
# ------------------------------------------------------------

question_files = {
    "train.csv": NEXTQA_QUESTIONS_DIR / "train.csv",
    "val.csv": NEXTQA_QUESTIONS_DIR / "val.csv",
    "test.csv": NEXTQA_QUESTIONS_DIR / "test.csv",
}

question_counts = {}

for split_name, file_path in question_files.items():

    question_counts[split_name] = len(
        pd.read_csv(file_path)
    )

total_questions = sum(question_counts.values())

# ------------------------------------------------------------
# Collect Metadata Resource Information
# ------------------------------------------------------------

metadata_files = sorted(
    file_path for file_path in NEXTQA_METADATA_DIR.iterdir()
    if file_path.is_file()
)

# ------------------------------------------------------------
# Display Final Verification Summary
# ------------------------------------------------------------

print("NExT-QA Dataset Preparation Summary")
print("=" * 45)

print("\nDataset Directories")
print("-" * 45)
print(f"Dataset directory:   {NEXTQA_DATASET_DIR}")
print(f"Videos directory:    {NEXTQA_VIDEOS_DIR}")
print(f"Questions directory: {NEXTQA_QUESTIONS_DIR}")
print(f"Metadata directory:  {NEXTQA_METADATA_DIR}")

print("\nVideo Dataset")
print("-" * 45)
print(f"Video files found:    {len(video_files)}")
print(f"Video subdirectories: {len(video_subdirs)}")
print(f"Extracted video size: {total_video_size_gb:.2f} GB")

print("\nQuestion-Answer Annotations")
print("-" * 45)

for split_name, count in question_counts.items():

    print(f"{split_name:<10}: {count:>8} rows")

print(f"{'Total':<10}: {total_questions:>8} rows")

print("\nMetadata Resources")
print("-" * 45)

for file_path in metadata_files:

    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"{file_path.name:<32} "
        f"{file_size_mb:>8.2f} MB"
    )

print("\nReadiness Check")
print("-" * 45)

if video_files and total_questions > 0 and metadata_files:

    print("Status: READY")
    print("Notebook 01 completed successfully.")
    print("NExT-QA dataset resources are available for downstream notebooks.")

else:

    print("Status: INCOMPLETE")
    print("One or more required dataset resources are missing.")

